# DiscoverBeer — Beer Recommendation System
ARTI 405 Project 2  
Team members: Garrett, Ferris, Colton, Ryan

# Part 1 — Data Preparation
**Assigned to: Garrett**

## 1.1 Import Libraries and Set Shared Settings

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 1.2 Load and Explore the RateBeer Dataset

Loading the dataset, displaying first 5 rows, and checking how many rows and columns are contained in the dataset.

In [2]:
# Load the dataset
df = pd.read_csv("ratebeer_for_model.xls", encoding="utf-8-sig")

# Display the first five rows
display(df.head())

# Show the number of rows and columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Check the data types
print(df.dtypes)

C:\Users\fords\AppData\Local\Temp\ipykernel_39440\2001076028.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("ratebeer_for_model.xls", encoding="utf-8-sig")


,beer/beerId,beer/name,beer/style,beer/ABV,review/profileName,review/overall
0,63836,John Harvards Simcoe IPA,India Pale Ale &#40;IPA&#41;,5.4,hopdog,13/20
1,63836,John Harvards Simcoe IPA,India Pale Ale &#40;IPA&#41;,5.4,TomDecapolis,13/20
2,71716,John Harvards Cristal Pilsner,Bohemian Pilsener,5,PhillyBeer2112,14/20
3,64125,John Harvards Fancy Lawnmower Beer,Klsch,5.4,TomDecapolis,8/20
4,64125,John Harvards Fancy Lawnmower Beer,Klsch,5.4,hopdog,8/20


Rows: 2924164
Columns: 6
beer/beerId           object
beer/name             object
beer/style            object
beer/ABV              object
review/profileName    object
review/overall        object
dtype: object


## 1.3 Clean the Data and Convert Ratings

In [3]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Check for dashes used as missing values
print("\nDash values:")
print((df == "-").sum())

# Count completely identical rows
print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
beer/beerId           1
beer/name             1
beer/style            1
beer/ABV              1
review/profileName    1
review/overall        1
dtype: int64

Dash values:
beer/beerId                0
beer/name                  0
beer/style                 0
beer/ABV              138637
review/profileName         0
review/overall             0
dtype: int64

Duplicate rows: 55260


*Observations*
- One missing value in each column
- There are 138,637 dash values in the ABV meaning there is a good handful of beers without a ABV however that isn't necessarily a problem because the ABV is just for more information for the user and not needed to calculate the recommended beer
- There are duplicated rows that need to be dropped

In [4]:
# Count rows missing at least one required value
rows_to_drop = df[[
    "beer/beerId",
    "review/profileName",
    "review/overall"
]].isnull().any(axis=1).sum()

print("Rows to drop:", rows_to_drop)

Rows to drop: 1


In [5]:
# Remove completely identical rows
df = df.drop_duplicates()

# Remove reviews missing information needed by the model
df = df.dropna(subset=[
    "beer/beerId",
    "review/profileName",
    "review/overall"
])

print("Rows remaining:", len(df))

Rows remaining: 2868903


Before removing rows that don't have required information, we wanted to see how many rows we would be getting rid of and if we should investigate further to maybe fill in those values; however, there is only one row, so we will drop it. We've also dropped all the duplicates.

In [6]:
# Store IDs and names as text
df["beer/beerId"] = df["beer/beerId"].astype("string")
df["beer/name"] = df["beer/name"].astype("string")
df["review/profileName"] = df["review/profileName"].astype("string")

# Store repeated styles as categories
df["beer/style"] = df["beer/style"].astype("category")

# Convert ABV to numbers; invalid values become missing
df["beer/ABV"] = pd.to_numeric(df["beer/ABV"], errors="coerce")

print(df.dtypes)

beer/beerId           string[python]
beer/name             string[python]
beer/style                  category
beer/ABV                     float64
review/profileName    string[python]
review/overall                object
dtype: object


This converts the existing data types, which were all labeled as objects, to more fitting data types. This allows inputs to come in correctly and be represented correctly. For the ABV column, if there is a "-" it will be converted to NaN.

In [8]:
# Extract the score before the slash and convert it to a number
rating_20 = pd.to_numeric(
    df["review/overall"].str.split("/").str[0],
    errors="coerce"
)

# Convert the 1–20 scale to a 1–5 scale
df["rating"] = 1 + (rating_20 - 1) * 4 / 19

# Compare the original and converted ratings
display(df[["review/overall", "rating"]].head())

print("Missing converted ratings:", df["rating"].isnull().sum())

,review/overall,rating
0,13/20,3.526316
1,13/20,3.526316
2,14/20,3.736842
3,8/20,2.473684
4,8/20,2.473684


Missing converted ratings: 0


In [9]:
# Check the range and distribution of the converted ratings
print(df["rating"].describe())

count    2.868903e+06
mean     3.575465e+00
std      7.004652e-01
min      1.000000e+00
25%      3.315789e+00
50%      3.736842e+00
75%      3.947368e+00
max      5.000000e+00
Name: rating, dtype: float64


This turns the rating into a number by creating a new rating column and converting the original review to a decimal rating instead of text. It also allows us to use our scale for the app which is 1-5.

## 1.4 Filter Beers and Reviewers with Too Few Ratings
We will check how much rating history each reviewer and beer has before deciding what to filter.

In [10]:
# Count different beers rated by each reviewer
reviewer_counts = df.groupby("review/profileName")["beer/beerId"].nunique()

# Count different reviewers for each beer
beer_counts = df.groupby("beer/beerId")["review/profileName"].nunique()

print("Different beers rated per reviewer:")
print(reviewer_counts.describe())

print("\nDifferent reviewers per beer:")
print(beer_counts.describe())

Different beers rated per reviewer:
count    29265.000000
mean        97.568768
std        454.737907
min          1.000000
25%          1.000000
50%          3.000000
75%         16.000000
max      16018.000000
Name: beer/beerId, dtype: float64

Different reviewers per beer:
count    110381.000000
mean         25.868129
std         109.749585
min           1.000000
25%           2.000000
50%           4.000000
75%          13.000000
max        3696.000000
Name: review/profileName, dtype: float64


*Observations*
- Half of reviewers rated 3 beers or fewer
- Half of beers have 4 reviewers or fewer

To resolve I want to keep the reviewers who rated at least 10 different beers so that gives us more history to learn from.

In [11]:
# Find reviewers who rated at least 10 different beers
keep_reviewers = reviewer_counts[reviewer_counts >= 10].index

# Keep their reviews
rows_before = len(df)
df = df[df["review/profileName"].isin(keep_reviewers)].copy()

print("Rows removed:", rows_before - len(df))
print("Rows remaining:", len(df))
print("Reviewers remaining:", df["review/profileName"].nunique())

Rows removed: 51079
Rows remaining: 2817824
Reviewers remaining: 9233


Now we have 2,817,824 reviews from 9,233 reviewers that have all reviewed at least 10 different beers.

## 1.5 Create Training, Validation, and Test Groups
We will split reviewers into 70% training, 15% validation, and 15% testing. Each reviewer will belong to only one group.

In [12]:
# Get the list of reviewers
reviewers = df["review/profileName"].unique()

# Split reviewers into 70% training and 30% remaining
train_users, remaining_users = train_test_split(
    reviewers, test_size=0.30, random_state=42
)

# Split the remaining reviewers equally into validation and testing
val_users, test_users = train_test_split(
    remaining_users, test_size=0.50, random_state=42
)

# Create a table of reviews for each group
train_df = df[df["review/profileName"].isin(train_users)].copy()
val_df = df[df["review/profileName"].isin(val_users)].copy()
test_df = df[df["review/profileName"].isin(test_users)].copy()

print("Training reviewers:", len(train_users))
print("Validation reviewers:", len(val_users))
print("Testing reviewers:", len(test_users))

Training reviewers: 6463
Validation reviewers: 1385
Testing reviewers: 1385


In [13]:
# Count different training reviewers for each beer
train_beer_counts = train_df.groupby(
    "beer/beerId"
)["review/profileName"].nunique()

# Find beers with at least 20 training reviewers
keep_beers = train_beer_counts[train_beer_counts >= 20].index

# Keep those beers in all three groups
train_df = train_df[train_df["beer/beerId"].isin(keep_beers)].copy()
val_df = val_df[val_df["beer/beerId"].isin(keep_beers)].copy()
test_df = test_df[test_df["beer/beerId"].isin(keep_beers)].copy()

print("Beers remaining:", len(keep_beers))
print("Training reviews:", len(train_df))
print("Validation reviews:", len(val_df))
print("Testing reviews:", len(test_df))

Beers remaining: 14563
Training reviews: 1552399
Validation reviews: 335727
Testing reviews: 363042


In [14]:
# Count extra reviews from the same person for the same beer
repeated_pairs = train_df.duplicated(subset=[
    "review/profileName",
    "beer/beerId"
]).sum()

print("Repeated reviewer/beer pairs:", repeated_pairs)

Repeated reviewer/beer pairs: 7299


*Observations*
- There are 7,299 extra training review rows where the same person rated the same beer more than once.
- Fix: We are going to average the multiple ratings from the same person and use that rating
- For example: Garrett rated Busch a 2/5, but there is another rating from Garrett on Busch with a rating of 4/5 so we will combine them and the rating for Garrett on Busch is 3/5.

In [15]:
# Columns that identify one reviewer/beer pair
pair_columns = ["review/profileName", "beer/beerId"]

# Create one rating per reviewer and beer
train_ratings = train_df.groupby(
    pair_columns, as_index=False
)["rating"].mean()

val_ratings = val_df.groupby(
    pair_columns, as_index=False
)["rating"].mean()

test_ratings = test_df.groupby(
    pair_columns, as_index=False
)["rating"].mean()

print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(val_ratings))
print("Testing ratings:", len(test_ratings))

display(train_ratings.head())

Training ratings: 1545100
Validation ratings: 334058
Testing ratings: 361334


,review/profileName,beer/beerId,rating
0,007lund,1092,2.684211
1,007lund,1094,4.368421
2,007lund,11498,4.157895
3,007lund,1217,4.578947
4,007lund,1221,4.157895


In [16]:
# Count ratings remaining for each validation and test reviewer
val_counts = val_ratings["review/profileName"].value_counts()
test_counts = test_ratings["review/profileName"].value_counts()

print("Validation reviewers with fewer than 6 ratings:",
      (val_counts < 6).sum())

print("Test reviewers with fewer than 6 ratings:",
      (test_counts < 6).sum())

Validation reviewers with fewer than 6 ratings: 6
Test reviewers with fewer than 6 ratings: 7


We need more than five ratings per test reviewer so we can give the model five as inputs and use the remaining hidden ratings to check whether it recommends beers they liked.

In [17]:
# Find reviewers with at least 6 ratings
keep_val_users = val_counts[val_counts >= 6].index
keep_test_users = test_counts[test_counts >= 6].index

# Keep their ratings
val_ratings = val_ratings[
    val_ratings["review/profileName"].isin(keep_val_users)
].copy()

test_ratings = test_ratings[
    test_ratings["review/profileName"].isin(keep_test_users)
].copy()

print("Validation reviewers remaining:",
      val_ratings["review/profileName"].nunique())

print("Test reviewers remaining:",
      test_ratings["review/profileName"].nunique())

Validation reviewers remaining: 1378
Test reviewers remaining: 1377


## 1.6 Build the Sparse Rating Matrix and Beer Catalog
We will organize the training ratings into a matrix with one row per reviewer and one column per beer.

In [18]:
from sklearn.preprocessing import LabelEncoder

# Create an encoder for reviewers and another for beers
user_encoder = LabelEncoder()
beer_encoder = LabelEncoder()

# Assign consecutive numbers starting at 0
train_ratings["user_index"] = user_encoder.fit_transform(
    train_ratings["review/profileName"]
)

train_ratings["beer_index"] = beer_encoder.fit_transform(
    train_ratings["beer/beerId"]
)

display(train_ratings.head())

,review/profileName,beer/beerId,rating,user_index,beer_index
0,007lund,1092,2.684211,0,856
1,007lund,1094,4.368421,0,870
2,007lund,11498,4.157895,0,1371
3,007lund,1217,4.578947,0,1915
4,007lund,1221,4.157895,0,1950


In [19]:
from scipy.sparse import csr_matrix

# Build the matrix using ratings and their row/column positions
train_matrix = csr_matrix(
    (
        train_ratings["rating"].to_numpy(),
        (
            train_ratings["user_index"].to_numpy(),
            train_ratings["beer_index"].to_numpy()
        )
    ),
    shape=(
        len(user_encoder.classes_),
        len(beer_encoder.classes_)
    ),
    dtype=np.float32
)

print("Matrix shape:", train_matrix.shape)
print("Stored ratings:", train_matrix.nnz)

Matrix shape: (6462, 14563)
Stored ratings: 1545100


In [20]:
# Keep one catalog entry per beer
beer_catalog = train_df[[
    "beer/beerId",
    "beer/name",
    "beer/style",
    "beer/ABV"
]].drop_duplicates(subset="beer/beerId").copy()

# Connect each beer to its matrix column
beer_catalog["beer_index"] = beer_encoder.transform(
    beer_catalog["beer/beerId"]
)

# Arrange the catalog in matrix-column order
beer_catalog = beer_catalog.sort_values(
    "beer_index"
).reset_index(drop=True)

display(beer_catalog.head())

,beer/beerId,beer/name,beer/style,beer/ABV,beer_index
0,1,Beamish Irish Stout,Dry Stout,4.1,0
1,100,Nutfield Auburn Ale,Irish Ale,4.6,1
2,10000,Dithmarscher Pilsener,Classic German Pilsener,4.8,2
3,100016,Braunstein Nsgaarden kologisk Pske,Dunkler Bock,6.5,3
4,10002,Dithmarscher Urtyp,Classic German Pilsener,4.9,4


When the model recommends a beer, the app uses its beer_index to look up the name and details in this catalog. We’ll save the catalog alongside the model so users see beer names, while the code uses the matching IDs and column numbers.

In [21]:
# Use the existing beer mapping for validation and test data
val_ratings["beer_index"] = beer_encoder.transform(
    val_ratings["beer/beerId"]
)

test_ratings["beer_index"] = beer_encoder.transform(
    test_ratings["beer/beerId"]
)

display(val_ratings.head())

,review/profileName,beer/beerId,rating,beer_index
0,10%drunk,11311,4.578947,1195
1,10%drunk,11327,4.789474,1220
2,10%drunk,12611,4.578947,2273
3,10%drunk,16654,4.578947,4221
4,10%drunk,22904,2.894737,5273


We use .transform(), rather than fitting another encoder, because each beer must have the same number in every group.

In [22]:
from scipy.sparse import save_npz
import joblib

# Save the sparse training matrix
save_npz("train_matrix.npz", train_matrix)

# Save the beer names and details
beer_catalog.to_csv("beer_catalog.csv", index=False)

# Save the mapping between beer IDs and column numbers
joblib.dump(beer_encoder, "beer_encoder.joblib")

# Save the rating tables as compressed CSV files
train_ratings.to_csv("train_ratings.csv.gz", index=False)
val_ratings.to_csv("val_ratings.csv.gz", index=False)
test_ratings.to_csv("test_ratings.csv.gz", index=False)

print("Prepared files saved.")

Prepared files saved.


In Part 1, we prepared the RateBeer dataset for model development. We started by checking for missing values and duplicates, removing repeated rows and reviews missing information needed by the model. We then corrected the data types and converted the overall ratings from a 1–20 scale to the 1–5 scale our app will use. Next, we kept reviewers who rated at least 10 different beers and split the reviewers into 70% training, 15% validation, and 15% testing. We kept beers with at least 20 training reviewers and averaged ratings when the same person reviewed the same beer more than once. For validation and testing, we kept reviewers with at least six ratings so we can use five as inputs and check recommendations against the remaining ratings. Finally, we assigned numeric indexes, created the sparse training matrix, and built a separate beer catalog containing names, styles, and ABV values. 

# Part 2 — Model Development
**Assigned to: __________**

*Make sure to run existing code*

## 2.1 Train the Matrix Factorization Model

## 2.2 Compare Model Settings Using Validation Results

## 2.3 Save and Load the Trained Model

## 2.4 Estimate Preferences from a New User's Ratings

## 2.5 Predict Ratings and Return the Top Five Recommendations

# Part 3 — Testing and Evaluation
**Assigned to: __________**

## 3.1 Prepare Test Users and Hide Ratings

## 3.2 Evaluate the Top Five Recommendations

## 3.3 Compare Against a Popular-Beer Baseline

## 3.4 Test Input Validation and Recommendation Outputs

## 3.5 Summarize Results, Bias, and Model Limitations